# Traffic Sign Detection & Speed Limit Identification — Training & Evaluation

**SDAIA Academy · Computer Vision Systems Development · Final Project**

Runs **locally on CPU** (no GPU needed). Start Jupyter with `caffeinate -i` so the Mac does not sleep during training.

| Step | What happens |
|---|---|
| 1. Setup | Paths and settings |
| 2. Dataset audit | Download the Kaggle dataset and find its problems (label noise, mirrored copies, leakage) |
| 3. Dataset preparation | Clean, re-map to 9 classes, split train / val / test by original photo |
| 4. Training | Fine-tune a pretrained **YOLO11n** (transfer learning) with augmentation |
| 5. Evaluation report | Precision, Recall, mAP@0.5, mAP@0.5:0.95, per-class table, confusion matrix, PR curve |
| 6. Success & failure cases | Compare predictions with ground truth on test images |
| 7. Deployment | Export to **ONNX**, compare speed & accuracy vs PyTorch |

## 1. Setup

In [ ]:
import os, time, shutil, random, collections
from pathlib import Path

import cv2
import numpy as np
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage
from ultralytics import YOLO

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'datasets' / 'traffic_signs'     # cleaned dataset is written here
RUNS = ROOT / 'runs'
RESULTS = ROOT / 'results'
WEIGHTS = ROOT / 'weights'
for d in (RESULTS, WEIGHTS):
    d.mkdir(exist_ok=True)

EPOCHS = 50          # ~7 min/epoch on an i7 CPU at 960 px -> run overnight (early stopping may finish sooner)
IMGSZ = 960          # use 640 for a ~2.5x faster run (less accurate on small signs); signs are small in 1360x800 photos; 960 keeps the digits readable
BATCH = 8
RUN_NAME = 'yolo11n_signs'
random.seed(0)

## 2. Dataset audit

**Source:** [Traffic Sign Detection Dataset (Kaggle)](https://www.kaggle.com/datasets/icebearogo/traffic-sign-detection-dataset)
— YOLO-format labels (`class cx cy w h`), 43 class IDs, **no class-name file**.
The IDs follow the German traffic sign standard (GTSRB), which we verified by looking at crops of every class.

The raw data has three problems that would make any result misleading, so we check each one before training.

In [ ]:
import kagglehub
SRC = Path(kagglehub.dataset_download('icebearogo/traffic-sign-detection-dataset')) / 'Traffic_sign_detection_data'

GTSRB = ['Speed limit 20', 'Speed limit 30', 'Speed limit 50', 'Speed limit 60', 'Speed limit 70',
         'Speed limit 80', 'End of speed limit 80', 'Speed limit 100', 'Speed limit 120', 'No passing',
         'No passing for trucks', 'Right-of-way at next intersection', 'Priority road', 'Yield', 'Stop',
         'No vehicles', 'No trucks', 'No entry', 'General caution', 'Dangerous curve left',
         'Dangerous curve right', 'Double curve', 'Bumpy road', 'Slippery road', 'Road narrows on the right',
         'Road work', 'Traffic signals', 'Pedestrians', 'Children crossing', 'Bicycles crossing',
         'Beware of ice/snow', 'Wild animals crossing', 'End of all limits', 'Turn right ahead',
         'Turn left ahead', 'Ahead only', 'Go straight or right', 'Go straight or left', 'Keep right',
         'Keep left', 'Roundabout', 'End of no passing', 'End of no passing for trucks']

def read_labels(lp):
    return [(int(float(r[0])), *map(float, r[1:5])) for r in (l.split() for l in lp.read_text().splitlines()) if r]

raw = []   # one row per raw image
for split in ['train', 'val']:
    for lp in sorted((SRC / 'labels' / split).glob('*.txt')):
        boxes = read_labels(lp)
        raw.append({'split': split, 'stem': lp.stem, 'photo': lp.stem.split('_')[0], 'copy': '_' in lp.stem,
                    'img': SRC / 'images' / split / f'{lp.stem}.jpg', 'boxes': boxes,
                    'only_class0': {b[0] for b in boxes} == {0}})
raw = pd.DataFrame(raw)
print(f"{len(raw)} raw images, {raw['boxes'].map(len).sum()} boxes, {raw['photo'].nunique()} distinct photo IDs")
raw.groupby(['split', 'copy']).size().rename('images').to_frame()

### Problem 1 — label noise: class 0 is used for *every* sign in part of the data
Class 0 should be "Speed limit 20" (one of the rarest signs), yet it has more boxes than all other classes combined.
Crops from images whose only label is class 0 show every kind of sign: that subset was annotated as a generic "sign".

In [ ]:
only0 = raw[raw['only_class0']]
print(f"{len(only0)} images ({len(only0) / len(raw):.0%}) contain only class-0 boxes")

def crop(img_path, box, pad=6, size=90):
    img = cv2.imread(str(img_path)); H, W = img.shape[:2]
    _, cx, cy, w, h = box
    x1, y1 = max(int((cx - w / 2) * W) - pad, 0), max(int((cy - h / 2) * H) - pad, 0)
    x2, y2 = int((cx + w / 2) * W) + pad, int((cy + h / 2) * H) + pad
    return cv2.resize(cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2RGB), (size, size))

sample = only0.sample(16, random_state=0)
plt.figure(figsize=(16, 2.4))
plt.imshow(np.hstack([crop(r.img, r.boxes[0]) for r in sample.itertuples()])); plt.axis('off')
plt.title('All labelled "class 0" — clearly different signs'); plt.savefig(RESULTS / 'audit_class0_noise.png', dpi=110, bbox_inches='tight'); plt.show()

### Problem 2 — mirrored copies
Every photo appears up to 6 times (`00000`, `00000_0` … `00000_4`). Several copies are **horizontally flipped**:
a mirrored "30" is not a real sign, and "turn left" becomes a picture of "turn right" while keeping the left label.

In [ ]:
def is_mirrored(orig, copy):
    a, b = cv2.imread(str(orig), 0), cv2.imread(str(copy), 0)
    if a.shape != b.shape:
        return None
    return np.abs(a[:, ::-1].astype(int) - b).mean() < np.abs(a.astype(int) - b).mean()

originals = raw[~raw['copy']].drop_duplicates('photo').set_index('photo')
copies = raw[raw['copy'] & raw['photo'].isin(originals.index)]
checked = [is_mirrored(originals.loc[r.photo, 'img'], r.img) for r in copies.sample(300, random_state=0).itertuples()]
checked = [c for c in checked if c is not None]
print(f'{sum(checked) / len(checked):.0%} of {len(checked)} sampled copies are mirror images of the original')

ex = raw[raw['photo'] == '00000'].sort_values('stem')
fig, axes = plt.subplots(1, len(ex), figsize=(4 * len(ex), 3))
for ax, r in zip(axes, ex.itertuples()):
    ax.imshow(cv2.cvtColor(cv2.imread(str(r.img)), cv2.COLOR_BGR2RGB)); ax.set_title(f'{r.stem} ({r.split})'); ax.axis('off')
plt.tight_layout(); plt.savefig(RESULTS / 'audit_mirrored_copies.png', dpi=110); plt.show()

### Problem 3 — train / validation leakage
Because the copies were split randomly, the *same photo* appears in both train and val,
so validation scores would measure memorisation rather than generalisation.

In [ ]:
leak = set(raw.loc[raw['split'] == 'train', 'photo']) & set(raw.loc[raw['split'] == 'val', 'photo'])
print(f"{len(leak)} of {raw.loc[raw['split'] == 'val', 'photo'].nunique()} validation photos also appear in train")

## 3. Dataset preparation
1. Keep **only original photos** (no copies) → removes mirrored images and duplicates.
2. Drop images labelled **only class 0** → removes the generic-label subset.
3. **Re-map to 9 classes** focused on the goal (speed-limit identification): the 7 speed limits with enough
   examples, *Stop*, and *Other Sign* for the remaining 35 sign types (including the single 20 km/h example).
4. New **70 / 15 / 15 split by photo** → no leakage.

In [ ]:
NAMES = ['Speed Limit 30', 'Speed Limit 50', 'Speed Limit 60', 'Speed Limit 70', 'Speed Limit 80',
         'Speed Limit 100', 'Speed Limit 120', 'Stop', 'Other Sign']
REMAP = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 7: 5, 8: 6, 14: 7}      # GTSRB id -> our id, anything else -> Other Sign
to_ours = lambda gid: REMAP.get(gid, NAMES.index('Other Sign'))
assert [NAMES[to_ours(g)] for g in (1, 8, 14, 0, 38)] == ['Speed Limit 30', 'Speed Limit 120', 'Stop', 'Other Sign', 'Other Sign']

clean = raw[~raw['copy'] & ~raw['only_class0']].drop_duplicates('photo').sample(frac=1, random_state=0).reset_index(drop=True)
n = len(clean)

# stratified split: group photos by the rarest class they contain, split each group 70/15/15,
# so every speed limit appears in val and test (a plain random split left test with no 60 km/h sign)
freq = collections.Counter(to_ours(b[0]) for bs in clean['boxes'] for b in bs)
clean['strata'] = clean['boxes'].map(lambda bs: min((to_ours(b[0]) for b in bs), key=freq.get))
clean['new_split'] = 'train'
for _, idx in clean.groupby('strata').groups.items():
    k = len(idx)
    n_val, n_test = max(round(k * .15), 1), max(round(k * .15), 1)
    clean.loc[idx[:n_test], 'new_split'] = 'test'
    clean.loc[idx[n_test:n_test + n_val], 'new_split'] = 'val'

shutil.rmtree(DATA, ignore_errors=True)
for r in clean.itertuples():
    for sub in ('images', 'labels'):
        (DATA / sub / r.new_split).mkdir(parents=True, exist_ok=True)
    shutil.copy(r.img, DATA / 'images' / r.new_split / r.img.name)
    (DATA / 'labels' / r.new_split / f'{r.stem}.txt').write_text(
        '\n'.join(f'{to_ours(c)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in r.boxes) + '\n')

DATA_YAML = DATA / 'data.yaml'
yaml.safe_dump({'path': str(DATA), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
                'names': dict(enumerate(NAMES))}, open(DATA_YAML, 'w'), sort_keys=False)

assert not (set(clean.loc[clean.new_split == 'train', 'photo']) & set(clean.loc[clean.new_split != 'train', 'photo']))
counts_df = pd.DataFrame({s: collections.Counter(to_ours(b[0]) for bs in clean.loc[clean.new_split == s, 'boxes'] for b in bs)
                          for s in ['train', 'val', 'test']}).reindex(range(len(NAMES))).fillna(0).astype(int)
counts_df.index = NAMES
print(f"clean dataset: {n} photos -> {dict(clean['new_split'].value_counts())}")
counts_df

### Exploratory data analysis
Box size matters: signs cover a tiny part of a 1360×800 photo, which is why we train at 960 px instead of 640.

In [ ]:
areas = [w * h for bs in clean['boxes'] for _, _, _, w, h in bs]
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
counts_df.plot.barh(ax=ax[0], stacked=True); ax[0].set_title('Instances per class (after cleaning)'); ax[0].set_xlabel('boxes')
ax[1].hist([a * 100 for a in areas], bins=40); ax[1].set_title('Box size (% of image area)'); ax[1].set_xlabel('% of image')
plt.tight_layout(); plt.savefig(RESULTS / 'dataset_eda.png', dpi=120); plt.show()
print(f'median sign size: {np.median(areas) * 100:.3f}% of the image '
      f'(≈ {np.sqrt(np.median(areas) * 1360 * 800):.0f} px wide in the original photo)')

In [ ]:
def draw_boxes(img, boxes, color, with_label=True):
    """boxes: list of (cls, [x1,y1,x2,y2] normalised, conf or None)"""
    h, w = img.shape[:2]
    for cls, (x1, y1, x2, y2), conf in boxes:
        p1, p2 = (int(x1 * w), int(y1 * h)), (int(x2 * w), int(y2 * h))
        cv2.rectangle(img, p1, p2, color, 2)
        if with_label:
            text = NAMES[cls] + (f' {conf:.2f}' if conf is not None else '')
            cv2.putText(img, text, (p1[0], max(p1[1] - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return img

def load_gt(img_path):
    lp = DATA / 'labels' / Path(img_path).parent.name / f'{Path(img_path).stem}.txt'
    return [(c, [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], None) for c, cx, cy, w, h in read_labels(lp)]

def list_images(split):
    return sorted((DATA / 'images' / split).glob('*.jpg'))

fig, axes = plt.subplots(2, 3, figsize=(20, 8))
for ax, p in zip(axes.flat, random.sample(list_images('train'), 6)):
    ax.imshow(draw_boxes(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB), load_gt(p), (0, 255, 0))); ax.axis('off')
plt.suptitle('Training samples with ground-truth boxes'); plt.tight_layout(); plt.show()

## 4. Training — transfer learning with YOLO11n

* **Model:** YOLO11n pretrained on COCO → new 9-class head, all layers fine-tuned. The nano model is small enough
  to train on a laptop CPU and to run in real time.
* **Preprocessing:** letterbox resize to 960×960, pixel values scaled to 0–1 (done by Ultralytics).
* **Augmentation (on the fly):**
  * mosaic (4 photos combined), random scale ±50 %, translate ±10 %
  * HSV colour jitter (lighting / weather)
  * **horizontal flip disabled (`fliplr=0`)** — the exact problem found in the raw data.
* If training is interrupted, re-run this cell: it **resumes** from the last checkpoint.

In [ ]:
RUN_DIR = RUNS / RUN_NAME
last = RUN_DIR / 'weights' / 'last.pt'

if last.exists():
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:          # raised when the run already finished
        print('Training already complete:', e)
else:
    YOLO('yolo11n.pt').train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=20, seed=0,
        device='cpu', workers=4, project=str(RUNS), name=RUN_NAME, exist_ok=True, plots=True,
        # augmentation
        mosaic=1.0, scale=0.5, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, fliplr=0.0,
    )

BEST = RUN_DIR / 'weights' / 'best.pt'
shutil.copy(BEST, WEIGHTS / 'best.pt')
display(IPImage(filename=str(RUN_DIR / 'results.png'), width=1100))

## 5. Evaluation report
Evaluated on the held-out **test** split (photos never seen during training or model selection).

| Metric | Meaning |
|---|---|
| Precision | Of the signs the model reported, how many were correct |
| Recall | Of the real signs in the images, how many the model found |
| mAP@0.5 | Mean average precision; a box counts as correct at IoU ≥ 0.5 |
| mAP@0.5:0.95 | Stricter: averaged over IoU 0.5 → 0.95 (box must be tight) |

In [ ]:
model = YOLO(str(BEST))
m = model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device='cpu', plots=True,
              project=str(RUNS), name='eval_test', exist_ok=True)

summary = pd.DataFrame({
    'metric': ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95', 'Inference (ms/img, CPU)'],
    'value': [m.box.mp, m.box.mr, m.box.map50, m.box.map, m.speed['inference']],
}).round(3)

per_class = pd.DataFrame([
    dict(zip(['class', 'precision', 'recall', 'mAP@0.5', 'mAP@0.5:0.95'], [NAMES[c], *m.box.class_result(i)]))
    for i, c in enumerate(m.box.ap_class_index)
])
per_class.insert(1, 'test boxes', [counts_df.loc[c, 'test'] for c in per_class['class']])
per_class.insert(2, 'train boxes', [counts_df.loc[c, 'train'] for c in per_class['class']])
per_class = per_class.sort_values('mAP@0.5').round(3).reset_index(drop=True)

display(Markdown('### Overall')); display(summary)
display(Markdown('### Per class (worst first)')); display(per_class)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(per_class['class'], per_class['mAP@0.5'])
ax.set_xlim(0, 1); ax.set_xlabel('mAP@0.5'); ax.set_title('Per-class mAP@0.5 (test set)')
plt.tight_layout(); plt.savefig(RESULTS / 'per_class_map.png', dpi=120); plt.show()

for f in ['confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    p = Path(m.save_dir) / f
    if p.exists():
        display(Markdown(f'**{f}**')); display(IPImage(filename=str(p), width=800))
        shutil.copy(p, RESULTS / f)
shutil.copy(RUN_DIR / 'results.png', RESULTS / 'training_curves.png')

In [ ]:
report = f"""# Evaluation Report — YOLO11n traffic sign detector

Test split: **{len(list_images('test'))} photos** never used for training · image size {IMGSZ} · weights `best.pt`

## Overall
{summary.to_markdown(index=False)}

## Per class (worst first)
{per_class.to_markdown(index=False)}
"""
(RESULTS / 'eval_report.md').write_text(report)
summary.to_csv(RESULTS / 'metrics_summary.csv', index=False)
per_class.to_csv(RESULTS / 'metrics_per_class.csv', index=False)
print('saved', RESULTS / 'eval_report.md')

## 6. Success & failure cases
Each test image is compared with its ground truth: a prediction is correct when its box overlaps a real sign
with IoU ≥ 0.5 **and** has the same class. An image is a *success* when every sign is found with the right class
and there are no extra detections; otherwise it is a *failure* (missed sign, wrong class, or false positive).

Green = ground truth · Red = prediction

In [ ]:
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def judge(gt, preds, thr=0.5):
    """Return list of problems (empty list = success)."""
    problems, used = [], set()
    for g_cls, g_box, _ in gt:
        best_j, best_iou = None, thr
        for j, (p_cls, p_box, _) in enumerate(preds):
            if j not in used and iou(g_box, p_box) >= best_iou:
                best_j, best_iou = j, iou(g_box, p_box)
        if best_j is None:
            problems.append(f'missed {NAMES[g_cls]}')
        else:
            used.add(best_j)
            if preds[best_j][0] != g_cls:
                problems.append(f'{NAMES[g_cls]} → predicted {NAMES[preds[best_j][0]]}')
    problems += [f'false positive {NAMES[c]} ({s:.2f})' for j, (c, _, s) in enumerate(preds) if j not in used]
    return problems

# self-check
assert abs(iou([0, 0, 2, 2], [1, 1, 3, 3]) - 1 / 7) < 1e-9 and iou([0, 0, 1, 1], [2, 2, 3, 3]) == 0
assert judge([(0, [0, 0, .5, .5], None)], [(0, [0, 0, .5, .5], .9)]) == []
assert judge([(0, [0, 0, .5, .5], None)], [(1, [0, 0, .5, .5], .9)]) == [f'{NAMES[0]} → predicted {NAMES[1]}']
assert judge([(0, [0, 0, .5, .5], None)], []) == [f'missed {NAMES[0]}']

In [ ]:
successes, failures = [], []
for r in model.predict([str(p) for p in list_images('test')], conf=0.25, imgsz=IMGSZ, device='cpu', stream=True, verbose=False):
    preds = [(int(c), b.tolist(), float(s)) for c, b, s in zip(r.boxes.cls, r.boxes.xyxyn, r.boxes.conf)]
    gt = load_gt(r.path)
    problems = judge(gt, preds)
    (failures if problems else successes).append((r.path, gt, preds, problems))

total = len(successes) + len(failures)
print(f'{len(successes)}/{total} test photos fully correct ({len(successes) / total:.1%})')
kinds = pd.Series([p.split(' ')[0] if p.startswith(('missed', 'false')) else 'wrong class'
                   for *_, probs in failures for p in probs], dtype=str).value_counts()
display(kinds.rename('failure type count').to_frame())

In [ ]:
def show_cases(cases, title, fname, k=3):
    cases = cases[:k]
    if not cases:
        return print('none:', title)
    fig, axes = plt.subplots(1, len(cases), figsize=(7 * len(cases), 4.5), squeeze=False)
    for ax, (path, gt, preds, problems) in zip(axes[0], cases):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        img = draw_boxes(img, gt, (0, 255, 0), with_label=False)
        img = draw_boxes(img, preds, (255, 0, 0))
        ax.imshow(img); ax.axis('off')
        ax.set_title('\n'.join(problems[:3]) or 'all signs correct', fontsize=10)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(RESULTS / fname, dpi=120); plt.show()

# successes that contain a speed limit; wrong-class failures first (most interesting for a speed-limit reader)
successes.sort(key=lambda s: not any(NAMES[c].startswith('Speed') for c, _, _ in s[1]))
failures.sort(key=lambda f: not any('→' in p for p in f[3]))
show_cases(successes, 'Successful predictions', 'success_cases.png')
show_cases(failures, 'Failure cases', 'failure_cases.png')

### Extra: the dataset's unlabelled "Visual testing" photos
20 extra road photos shipped without labels — a quick visual check on unseen scenes.

In [ ]:
visual = sorted((SRC / 'Visual testing dataset').glob('*.png'))[:8]
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, r in zip(axes.flat, model.predict([str(p) for p in visual], conf=0.25, imgsz=IMGSZ, device='cpu', verbose=False)):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.savefig(RESULTS / 'visual_testing.png', dpi=110); plt.show()

### Analysis & how the model could be improved
*Fill in after looking at the results above.* Points to check:
* **Small dataset** (≈500 clean photos) → classes with the fewest training boxes should score lowest in the per-class table.
  More data would help most: e.g. re-label the 4K "class 0" subset with real classes.
* **Similar speed limits** (30 ↔ 80, 50 ↔ 60) → a second-stage classifier on the cropped sign trained on the
  large GTSRB crop dataset (50k images) would read digits far better.
* **Small / distant signs missed** → larger model (YOLO11s), higher resolution (1280), or tiling.
* **Domain gap** — German signs; Saudi signs have Arabic digits too → fine-tune on local footage.

## 7. Deployment — ONNX export
ONNX makes the model portable: it runs with `onnxruntime` on any CPU without PyTorch, which suits
in-car or edge devices. We compare accuracy and CPU speed of both formats.

In [ ]:
ONNX = Path(model.export(format='onnx', imgsz=IMGSZ, simplify=True))
shutil.copy(ONNX, WEIGHTS / 'best.onnx')

def cpu_ms_per_image(weights, imgs):
    mdl = YOLO(str(weights), task='detect')
    mdl.predict(str(imgs[0]), device='cpu', imgsz=IMGSZ, verbose=False)  # warm-up
    t = time.perf_counter()
    for p in imgs:
        mdl.predict(str(p), device='cpu', imgsz=IMGSZ, verbose=False)
    return (time.perf_counter() - t) / len(imgs) * 1000

bench = list_images('test')[:30]
m_onnx = YOLO(str(ONNX), task='detect').val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=1, device='cpu',
                                          project=str(RUNS), name='eval_onnx', exist_ok=True, plots=False)
deploy = pd.DataFrame([
    {'format': 'PyTorch (.pt)', 'size MB': BEST.stat().st_size / 1e6, 'mAP@0.5': m.box.map50, 'CPU ms/img': cpu_ms_per_image(BEST, bench)},
    {'format': 'ONNX (.onnx)', 'size MB': ONNX.stat().st_size / 1e6, 'mAP@0.5': m_onnx.box.map50, 'CPU ms/img': cpu_ms_per_image(ONNX, bench)},
]).round(3)
deploy['CPU FPS'] = (1000 / deploy['CPU ms/img']).round(1)
display(deploy)
with open(RESULTS / 'eval_report.md', 'a') as f:
    f.write(f'\n## Deployment (CPU, {len(bench)} images)\n{deploy.to_markdown(index=False)}\n')
print('weights saved to', WEIGHTS)